### **Lấy và làm sạch Dữ liệu AQI và Weather từ AQI**

![Ảnh](./images/pam1_JLLG.png)

#### **I. Import các thư viện**

In [1]:
import requests
import json
import pandas as pd
import numpy as np
import datetime
from datetime import datetime, timedelta
from time import sleep
%matplotlib inline      

#### **II. Lấy dữ liệu từ API và tạo Dataframe từ data lấy được**
Trong dự án này, dữ liệu chất lượng không khí được thu thập thông qua [Weatherbit API](https://www.weatherbit.io/), một nền tảng tổng hợp dữ liệu khí tượng – môi trường toàn cầu.Weatherbit không trực tiếp đặt cảm biến tại từng vị trí địa lý, mà tổng hợp dữ liệu từ nhiều nguồn khác nhau, bao gồm:
- Các trạm quan trắc mặt đất (ground-based stations),

- Dữ liệu vệ tinh khí tượng (NASA MODIS, TROPOMI, MOPITT, v.v.),

- Và các mô hình khí tượng – hóa học (numerical models) như WRF-Chem, CAMS, GEOS-Chem.

Dữ liệu sau khi được thu thập sẽ được hiệu chỉnh sai số, nội suy không gian và chuẩn hóa theo từng ô lưới (grid) có kích thước khoảng 10 km × 10 km.
Điều này có nghĩa là các khu vực nằm trong cùng một ô lưới (ví dụ các quận nội thành Hà Nội nằm gần nhau) sẽ nhận được cùng một giá trị AQI và nồng độ các chất ô nhiễm (PM₂.₅, PM₁₀, CO, NO₂, SO₂, O₃).

Vì vậy, để tránh hiện tượng dữ liệu bị trùng lặp (nhân bản) giữa các quận lân cận, dự án lựa chọn tọa độ trung tâm của quận Hoàn Kiếm (21.0285°N, 105.8542°E) làm điểm đại diện cho khu vực nội thành Hà Nội. Hoàn Kiếm là khu vực trung tâm thủ đô, có mật độ dân cư cao, nhiều hoạt động giao thông và thương mại, do đó phản ánh tương đối chính xác chất lượng không khí trung bình của toàn khu vực đô thị Hà Nội.


In [ ]:
API_KEY = "d0abdba555a24c308b658ff1a9af5267"
# API_KEY = "edb2db4cc2b84bec8a09cc37173ca0cc"
LAT, LON = 21.0285, 105.8542


start_date = datetime(2025, 1, 1)
end_date = datetime(2025, 1, 31, 0)

urls_air = []
urls_wea = []
current = start_date
while current < end_date:
    next_month = (current.replace(day=28) + timedelta(days=4)).replace(day=1)
    start_str = current.strftime('%Y-%m-%d')
    end_str = next_month.strftime('%Y-%m-%d')

    url_air = f"https://api.weatherbit.io/v2.0/history/airquality?lat={LAT}&lon={LON}&start_date={start_str}&end_date={end_str}&tz=local&key={API_KEY}"
    url_wea = f"https://api.weatherbit.io/v2.0/history/hourly?lat={LAT}&lon={LON}&start_date={start_str}&end_date={end_str}&tz=local&key={API_KEY}"
    urls_air.append(url_air)
    urls_wea.append(url_wea)
    current = next_month

print(f" Tạo{len(urls_air)} URLs ({urls_air[0]} → {urls_air[-1]})")
print(f" Tạo{len(urls_wea)} URLs ({urls_wea[0]} → {urls_wea[-1]})")


 Tạo1 URLs (https://api.weatherbit.io/v2.0/history/airquality?lat=21.0285&lon=105.8542&start_date=2025-01-01&end_date=2025-02-01&tz=local&key=d0abdba555a24c308b658ff1a9af5267 → https://api.weatherbit.io/v2.0/history/airquality?lat=21.0285&lon=105.8542&start_date=2025-01-01&end_date=2025-02-01&tz=local&key=d0abdba555a24c308b658ff1a9af5267)
 Tạo1 URLs (https://api.weatherbit.io/v2.0/history/hourly?lat=21.0285&lon=105.8542&start_date=2025-01-01&end_date=2025-02-01&tz=local&key=d0abdba555a24c308b658ff1a9af5267 → https://api.weatherbit.io/v2.0/history/hourly?lat=21.0285&lon=105.8542&start_date=2025-01-01&end_date=2025-02-01&tz=local&key=d0abdba555a24c308b658ff1a9af5267)


##### **2.1 AIR QUALITY**

In [8]:
results_air = []
for i, url in enumerate(urls_air):
    print(f'Lấy dữ liệu từ URL {i}/{len(urls_air)} : {url}')
    try:
        renponse = requests.get(url, timeout=30)
        renponse.raise_for_status()
        data = json.loads(renponse.text)
        results_air.append(data)
        sleep(1.2)

    except Exception as e:
        print(f"Lỗi khi lấy dữ liệu {i} : {e}")

print(f"\n Hoàn tất tải {len(results_air)} ")

Lấy dữ liệu từ URL 0/1 : https://api.weatherbit.io/v2.0/history/airquality?lat=21.0285&lon=105.8542&start_date=2025-01-01&end_date=2025-02-01&tz=local&key=d0abdba555a24c308b658ff1a9af5267

 Hoàn tất tải 1 


In [9]:
results_air[0]['city_name']

'Hoàn Kiếm'

In [10]:
results_air[0]['data'][0]

{'aqi': 74,
 'co': 10,
 'datetime': '2025-01-31:17',
 'no2': 22,
 'o3': 46.7,
 'pm10': 27,
 'pm25': 23,
 'so2': 40.3,
 'timestamp_local': '2025-02-01T00:00:00',
 'timestamp_utc': '2025-01-31T17:00:00',
 'ts': 1738342800}

In [11]:
joined_data = []
for res in results_air:
    if 'data' in res:
        joined_data.extend(res['data'])

joined_results = {
    'city_name': results_air[0]['city_name'],
    'country_code': results_air[0]['country_code'],
    'lat': results_air[0]['lat'],
    'lon': results_air[0]['lon'],
    'timezone': results_air[0]['timezone'],
    'data': joined_data
}


In [12]:
joined_results['data'][0]

{'aqi': 74,
 'co': 10,
 'datetime': '2025-01-31:17',
 'no2': 22,
 'o3': 46.7,
 'pm10': 27,
 'pm25': 23,
 'so2': 40.3,
 'timestamp_local': '2025-02-01T00:00:00',
 'timestamp_utc': '2025-01-31T17:00:00',
 'ts': 1738342800}

In [13]:
joined_results['data'][-1]

{'aqi': 155,
 'co': 1613,
 'datetime': '2024-12-31:17',
 'no2': 8,
 'o3': 2.2,
 'pm10': 174,
 'pm25': 155,
 'so2': 32,
 'timestamp_local': '2025-01-01T00:00:00',
 'timestamp_utc': '2024-12-31T17:00:00',
 'ts': 1735664400}

In [16]:

df = pd.DataFrame(joined_results)

df.columns = ['City', 'Country code', 'Lat', 'Lon', 'timezone', 'Data']
df[['AQI', 'CO', 'Date Time', 'NO2', 'O3', 'PM10', 'PM25', 'SO2', 'Local Time', 'UTC Time', 'TS']] = pd.DataFrame(df['Data'].tolist())
df.drop(columns=['Data', 'Lat', 'Lon', 'TS', 'Date Time'], inplace=True)
df = df.drop_duplicates()
df = df.sort_values(by='Local Time')
df['Local Time'] = pd.to_datetime(df['Local Time'])
df.set_index('Local Time', inplace=True)
df

,City,Country code,timezone,AQI,CO,NO2,O3,PM10,PM25,SO2,UTC Time
Local Time,,,,,,,,,,,
2025-01-01 00:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,155,1613.0,8.0,2.2,174.0,155.00,32.0,2024-12-31T17:00:00
2025-01-01 01:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,227,416.1,37.0,20.0,113.0,113.00,106.0,2024-12-31T18:00:00
2025-01-01 02:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,227,418.6,36.3,16.0,114.0,113.67,93.7,2024-12-31T19:00:00
2025-01-01 03:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,228,421.2,35.7,12.0,115.0,114.33,81.3,2024-12-31T20:00:00
2025-01-01 04:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,229,423.7,35.0,8.0,116.0,115.00,69.0,2024-12-31T21:00:00
...,...,...,...,...,...,...,...,...,...,...,...
2025-01-31 20:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,103,97.7,16.0,64.7,41.1,36.33,41.0,2025-01-31T13:00:00
2025-01-31 21:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,173,10.0,8.0,3.6,91.3,73.00,14.0,2025-01-31T14:00:00
2025-01-31 22:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,115,99.2,16.0,56.0,51.3,41.00,43.0,2025-01-31T15:00:00


In [17]:
df.to_csv('air_quality_data.csv', index=True)

In [18]:
df = pd.read_csv('air_quality_data.csv')
df.shape

(745, 12)

##### **2.2 WEATHER**

In [19]:
results_wea = []
for i, url in enumerate(urls_wea):
    print(f'Lấy dữ liệu từ URL {i}/{len(urls_wea)} : {url}')
    try:
        renponse = requests.get(url, timeout=30)
        renponse.raise_for_status()
        data = json.loads(renponse.text)
        results_wea.append(data)
        sleep(1.2)

    except Exception as e:
        print(f"Lỗi khi lấy dữ liệu {i} : {e}")

print(f"\n Hoàn tất tải {len(results_wea)} ")

Lấy dữ liệu từ URL 0/1 : https://api.weatherbit.io/v2.0/history/hourly?lat=21.0285&lon=105.8542&start_date=2025-01-01&end_date=2025-02-01&tz=local&key=d0abdba555a24c308b658ff1a9af5267

 Hoàn tất tải 1 


In [20]:
joined_data = []
for res in results_wea:
    if 'data' in res:
        joined_data.extend(res['data'])

joined_results_weather = {
    'city_name': results_wea[0]['city_name'],
    'country_code': results_wea[0]['country_code'],
    'lat': results_wea[0]['lat'],
    'lon': results_wea[0]['lon'],
    'timezone': results_wea[0]['timezone'],
    'data': joined_data
}

In [21]:

df_weather = pd.DataFrame(joined_results_weather)


df_weather.columns = ['City', 'Country code', 'Lat', 'Lon', 'timezone', 'Data']

d = pd.json_normalize(df_weather.pop('Data'))

keep = d[["clouds","precip","pres","rh","temp","uv","wind_spd","timestamp_local","timestamp_utc"]].rename(columns={
    "clouds":"Clouds",
    "precip":"Precipitation",
    "pres":"Pressure",
    "rh":"Relative Humidity",
    "temp":"Temperature",
    "uv":"UV Index",
    "wind_spd":"Wind Speed",
    "timestamp_local":"Local Time",
    "timestamp_utc":"UTC Time"
})
df_weather = pd.concat([df_weather[['City', 'Country code', 'timezone']], keep], axis=1)

df_weather.head()

,City,Country code,timezone,Clouds,Precipitation,Pressure,Relative Humidity,Temperature,UV Index,Wind Speed,Local Time,UTC Time
0,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,1,0.0,1018,95,13.6,0.0,1.6,2025-01-01T00:00:00,2024-12-31T17:00:00
1,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,0,0.0,1018,96,13.2,0.0,1.6,2025-01-01T01:00:00,2024-12-31T18:00:00
2,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,0,0.0,1017,96,13.1,0.0,2.0,2025-01-01T02:00:00,2024-12-31T19:00:00
3,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,0,0.0,1017,96,12.9,0.0,2.0,2025-01-01T03:00:00,2024-12-31T20:00:00
4,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,1,0.0,1017,97,12.8,0.0,1.6,2025-01-01T04:00:00,2024-12-31T21:00:00


In [22]:
df_weather.to_csv('weather_data.csv', index=False)

In [23]:
df_weather = pd.read_csv('weather_data.csv')
# df_weather.info()


#### **IV. Hợp nhất hai khung dữ liệu và sắp xếp dữ liệu**

In [24]:

merged_df = pd.merge(df, df_weather, left_index=True, right_index=True)

merged_df.drop(columns=['City_y', 'Country code_y', 'timezone_y', 'UTC Time_y'], inplace=True)

utc_time_column = merged_df.pop('UTC Time_x')
merged_df.insert(0, 'UTC Time', utc_time_column)

merged_df = merged_df.rename(columns={'City_x': 'City', 'Country code_x': 'Country Code', 'timezone_x':'Timezone'})
merged_df

,UTC Time,Local Time_x,City,Country Code,Timezone,AQI,CO,NO2,O3,PM10,PM25,SO2,Clouds,Precipitation,Pressure,Relative Humidity,Temperature,UV Index,Wind Speed,Local Time_y
0,2024-12-31T17:00:00,2025-01-01 00:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,155,1613.0,8.0,2.2,174.0,155.00,32.0,1,0.0,1018,95,13.6,0.0,1.60,2025-01-01T00:00:00
1,2024-12-31T18:00:00,2025-01-01 01:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,227,416.1,37.0,20.0,113.0,113.00,106.0,0,0.0,1018,96,13.2,0.0,1.60,2025-01-01T01:00:00
2,2024-12-31T19:00:00,2025-01-01 02:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,227,418.6,36.3,16.0,114.0,113.67,93.7,0,0.0,1017,96,13.1,0.0,2.00,2025-01-01T02:00:00
3,2024-12-31T20:00:00,2025-01-01 03:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,228,421.2,35.7,12.0,115.0,114.33,81.3,0,0.0,1017,96,12.9,0.0,2.00,2025-01-01T03:00:00
4,2024-12-31T21:00:00,2025-01-01 04:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,229,423.7,35.0,8.0,116.0,115.00,69.0,1,0.0,1017,97,12.8,0.0,1.60,2025-01-01T04:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
739,2025-01-31T12:00:00,2025-01-31 19:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,98,96.9,16.0,69.0,36.0,34.00,40.0,21,0.0,1009,72,21.8,0.0,2.00,2025-01-31T19:00:00
740,2025-01-31T13:00:00,2025-01-31 20:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,103,97.7,16.0,64.7,41.1,36.33,41.0,31,0.0,1010,76,21.1,0.0,1.66,2025-01-31T20:00:00
741,2025-01-31T14:00:00,2025-01-31 21:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,173,10.0,8.0,3.6,91.3,73.00,14.0,40,0.0,1010,81,20.4,0.0,1.33,2025-01-31T21:00:00
742,2025-01-31T15:00:00,2025-01-31 22:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,115,99.2,16.0,56.0,51.3,41.00,43.0,52,0.0,1011,85,19.7,0.0,1.00,2025-01-31T22:00:00


In [25]:

merged_df.shape

(744, 20)

In [26]:
merged_df.rename(columns={'Local Time_x': 'Local Time'}, inplace=True)
merged_df = merged_df[['Local Time', 'UTC Time', 'City', 'Country Code', 'Timezone', 'AQI', 'CO', 'NO2', 'O3', 'PM10', 'PM25', 'SO2',
                       'Clouds', 'Precipitation', 'Pressure', 'Relative Humidity', 'Temperature', 'UV Index', 'Wind Speed']]

merged_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 744 entries, 0 to 743
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Local Time         744 non-null    object 
 1   UTC Time           744 non-null    object 
 2   City               744 non-null    object 
 3   Country Code       744 non-null    object 
 4   Timezone           744 non-null    object 
 5   AQI                744 non-null    int64  
 6   CO                 744 non-null    float64
 7   NO2                744 non-null    float64
 8   O3                 744 non-null    float64
 9   PM10               744 non-null    float64
 10  PM25               744 non-null    float64
 11  SO2                744 non-null    float64
 12  Clouds             744 non-null    int64  
 13  Precipitation      744 non-null    float64
 14  Pressure           744 non-null    int64  
 15  Relative Humidity  744 non-null    int64  
 16  Temperature        744 non-null

In [27]:

merged_df.to_csv('hanoi-aqi-weather-data-TEST_1.csv', index=False)